#Original_Load_Preparation_Data

*Propósito:* Construir la tabla maestra de features (portafolio + variables) aplicando lógica de negocio: Dueños, materialidad, variables AG, antiguedad SUNAT,etc. Escritura particionada por codmes.

In [0]:
%run ../config/config

In [0]:
import sys
import time
import os
import numpy as np
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import IntegerType, DecimalType, DoubleType, StringType
from pyspark.sql.window import Window
from pyspark.sql.functions import udf, array

sys.path.insert(0, str(PROJECT_PATH))
from utils.logger import MLOpsLogger, log_dataframe_info
from utils.month_delta import month_delta

In [0]:
# Logger load preparation data

logger = MLOpsLogger(
    name='03_load_preparation_data',
    log_level='DEBUG' if DEBUG_MLOPS else 'INFO",
    log_dir=LOGS_PATH,
    is_job_run=True,
    job_context={
        'mes_vta': MES_VTA,
        'environment': ENV,
        'notebook': '03_load_preparation_data'
    }
)

In [0]:
def add_codmes_spark(col_name, months):
    return F.expr(f"""    
        cast(
            date_format(
                add_months(
                    to_date(cast({col_name} as string), 'yyyyMM'),
                    {months}
                ),
                'yyyymm'
            ) as int
        )
    """)

def operacionesMaxBetweenCols(columnas):
    """UDF para calcular el máximo entre columnas manejando nulos (versión reference)."""
    resultado = 0.0
    valorInicial = float(-99999999999999.0)
    mayor = float(0.0)
    numero = float(0.0)
    for column in columnas:
        if column is not None and str(column) != "" and str(column).upper() != "NULL":
            numero = float(column)
        else:
            numero = -99999999999999.0
        if numero >= valorInicial:
            mayor = float(numero)
            valorInicial = mayor
        elif numero < valorInicial:
            mayor = float(valorInicial)
    if mayor == -99999999999999.0:
        resultado = None
    else:
        resultado = mayor
    return resultado

operacionesMaxBetweenCols_udf = udf(operacionesMaxBetweenCols, DoubleType())

In [0]:
if RUN_HISTORICAL and MESES_HISTORICOS_CALIDAD > 0:
    meses_historicos_lista = []
    mes_tmp = MES_VTA
    for i in range(MESES_HISTORICOS_CALIDAD):
        mes_tmp = month_delta(str(mes_tmp), -1)
        meses_historicos_lista.append(mes_tmp)
    meses_historicos_lista.reverse()
    meses_a_procesar = meses_historicos_lista + [MES_VTA]
    logger.info(f"MODO HISTÓRICO: {len(meses_a_procesar)} meses: {meses_a_procesar}")

else:
    meses_a_procesar = [MES_VTA]
    logger.info(f"Procesando únicamente el mes actual: {MES_VTA}")

spark = SparkSession.builder.getOrCreate()
resultados = []

# Nombre de la tabla de salida (siguiendo el estándar del proyecto)
OUTPUT_FEATURES_TABLE = f"{CATALOG_NAME}.{SCHEMA_NAME}.{BASE_NAME_TABLE_MASTER}{ENV}"
logger.info(f"Tabla de salida: {OUTPUT_FEATURES_TABLE}")

# COMMAND ----

for idx_mes, mes_actual_proceso in enumerate(meses_a_procesar, 1):
    logger.info("")
    logger.info("=" * 80)
    logger.info(f"PROCESANDO MES {idx_mes}/{len(meses_a_procesar)}: {mes_actual_proceso}")
    logger.info("=" * 80)

    try:
        logger.log_stage_start('load_preparation_data', mes_vta-mes_actual_proceso, environment=ENV)
        start_time = time.time()

        # Mes de datos = 1 mes antes del mes proceso (desfase de 1 mes)
        mes_data = month_delta(str(mes_actual_proceso), -1)
        logger.info(f"Mes proceso: {mes_actual_proceso} | Mes data (desfase -1): {mes_data}")

        #=================================================
        # 1. UNIVERSO DE CLIENTES PYME (sin desfase, usa mes_actual_proceso)
        #=================================================
        logger.info("PASO 1: Universo de clientes PYME")
        sp_clientes = spark.table(SOURCE_TABLE).select(
            "codmes", "codclaveunicoclí", "codinternocomputacional", "codclavepartyccli"
        ).filter(
            F.trim(F.col("codproductonivelorbm")).isin("PYME REVOLVENTE", "PYME NO REVOLVENTE"),
            &(F.col("codmes") == mes_actual_proceso),
            &(F.col("codclaveunicoclí").isNotNull()),
            &(F.col("ctdmesmaduracion") > 0)
        ).distinct()

        universo = sp_clientes.groupBy("codclaveunicoclí", "codmes").agg(
            F.max("codinternocomputacional").alias("codinternocomputacional"),
            F.max("codclavepartyccli").alias("codclavepartyccli")
        )
        universe_count = universo.count()
        logger.info(f"Universo: {universe_count}, registros:")

        #=================================================
        # 2. VARIABLES DE DUEÑO Y RELACIÓN (desfase +1)
        #=================================================
        logger.info("PASO 2: Identificación de dueños y relación")
        cliente_prospecto = spark.table(
            "catalog_hlcl_prod_bcp.bcp_ddv_rbmbcapym_modelogestion_vu.hm_clienteprospectopyme"
        ).filter(F.col("codmes") == mes_data).select(
            F.col("CODMES").alias("CODMES_0"),
            "CODCLAVEUNICOCLI",
            "tippartyidentificacion"
        )
        cliente_prospecto = cliente_prospecto.withColumn("codmes", add_codmes_spark("CODMES_0", +1))
        cliente_prospecto = cliente_prospecto.drop("CODMES_0")

        relacionados = spark.table(
            "catalog_hlcl_prod_bcp.bcp_ddv_rbmbcapym_appyme_vu.hm_relacionadoapppyme"
        ).filter(F.col("codmes") == mes_data).select(
            F.col("CODMES").alias("CODMES_0"),
            "CODCLAVEUNICOCLI",
            "CODCLAVEUNICOCLIREL",
            "DESTIPREL"
        )
        relacionados = relacionados.withColumn("codmes", add_codmes_spark("CODMES_0", +1))
        relacionados = relacionados.drop("CODMES_0")

        relacionados = relacionados.join(
            cliente_prospecto,
            ["codmes", "CODCLAVEUNICOCLI"],
            "left outer"
        )
        
        # FIX 4: duenio_unic0 - PN usa CODLCAVEUNICOCLI, PJ usa CODLCAVEUNICOCLIREL
        duenios = relacionados.filter(
            F.col("DESTIPREL").isin("DUENIO", "DUENIO P.N.")
        ).select(
            "codmes", "CODLCAVEUNICOCLI", "CODLCAVEUNICOCLIREL",
            F.when(F.col("tippartydentificacion") == '6', "J").otherwise("N").alias("FLGTIPPER")
        )

        duenio_unic0 = duenios.groupBy("codmes", "CODLCAVEUNICOCLI").agg(
            F.max(
            F.when(F.col("FLGTIPPER") == "N", F.col("CODLCAVEUNICOCLI"))
            .otherwise(F.col("CODLCAVEUNICOCLIREL"))
            ).alias("CODLCAVEUNICOCLIREL")
        )

        relacionados_con_flag = relacionados.alias("A").join(
            duenio_unic0.alias("B"),
            (F.col("A.CODCLAVEUNICOCLI") == F.col("B.CODCLAVEUNICOCLI"))
            & (F.col("A.CODCLAVEUNICOCLIREL") == F.col("B.CODCLAVEUNICOCLIREL"))
            & (F.col("A.codmes") == F.col("B.codmes"))
            .left outer(),
        ).select(
            F.col("A.codmes"), F.col("A.CODCLAVEUNICOCLI"), F.col("A.CODCLAVEUNICOCLIREL"),
            F.when(
            F.col("A.DESTIPREL").isin("DUENIO", "DUENIO P.N."),
            F.when(F.col("B.CODCLAVEUNICOCLI").isNull(), O).otherwise(I)
            ).otherwise(O).alias("FLGRELUENIO")
        )

        relacion_dueno_final = relacionados.con_flag.filter(F.col("FLGRELUENIO") == 1).select(
            "codmes", "CODLCAVEUNICOCLI", "CODLCAVEUNICOCLIREL"
        )

        # ======================================================
        # 3. VARIABLES DEL CLIENTE (edad, actividad económica) (desfase +1)
        # ======================================================
        logger.info("PASO 3: Variables demográficas del cliente")
        universo_apppyme = spark.table(
            "catalog_lhcl_prod_bcp.bcp.ddv_rbmbcapym_apppyme_vu.hm_clientepreaprobacionapppyme",
            "filter(F.col('codmes') == mes_data).select(
            F.col('CODCMES').alias('CODCMES o'),
            "CODCLAVEUNICOCCI",
            F.col('NUMEDAD').cast(IntegerType()).alias("EDAD FIN"),
            F.col('DESSECCIONECONOMICAAPPPMYE').alias("ACT ECO FIN"),
            "TIPPARTYIDENTIFICACION"
        )
        universo_apppyme = universo_apppyme.withColumn('codmes', add_codmes_spark('CODCMES o', +1))
        universo_apppyme = universo_apppyme.drop('CODCMES o')

        # ======================================================
        # 4. VARIABLES DE CARRETERA (RCC, saldos, materialidad) (desfase +1)
        # ======================================================
        logger.info("PASO 4: Construcción de variables base (carretera)")

        # 4.1 Matriz de deuda RCC (base) - SIN columnas de meses_activo (se traen aparte)
        carretera_rcc = spark.table(
            "catalog_lhcl_prod_bcp.bcp_ddv_matrizvariables_vu.hm_matrizdeudorrccproducto"
        ).filter(F.col("codmes") == mes_data).select(
            F.col("CODMES").alias("CODMES_0"),
            "codclaveunicoclí",
            F.col("RCC_TIP_COND_MOR_MAX_CRNNR_MAX_U12").alias("ATRASOMAX_CRNENR_12"),
            F.col("RCC_MTO_ADE_ACT_SHIP_RT_U6M").alias("MONTOADE_ACT_MAX6_S_HIP"),
            F.col("RCC_PCT_UTL_3_RT_U3M").alias("UTL_3"),
            F.col("RCC_CTD_SF_CAL_CPP_FRQ_0_U24").alias("SF_NUM_CAL_CPP24")
        )
        carretera_rcc = carretera_rcc.withColumn('codmes', add_codmes_spark('CODMES_0', +1))
        carretera_rcc = carretera_rcc.drop("CODMES_0")

        # 4.2 Resumen saldo activo/pasivo
        resumen_saldo = spark.table(
            "catalog_lhcl_prod_bcp.bcp_ddv_matrizvariables_vu.hm_matrizresumensaldoactivopasivo"
        ).filter(F.col("codmes") == mes_data).select(
            F.col("CODMES").alias("CODMES_0"),
            "codclaveunicoclí",
            F.col("PROD_MTO_SLD_PRM_TSAV_FRQ_100_U24").alias("MESES_PMSAVMF_24_100")
        )
        resumen_saldo = resumen_saldo.withColumn('codmes', add_codmes_spark('CODMES_0', +1))
        resumen_saldo = resumen_saldo.drop("CODMES_0")

        # 4.3 Variables de materialidad (ATRASOMAX, SF_NUM_CAL_CPP24, MESES_ACTIVO)
        # FIX 2: Agregar RCC_CTD_MES_ACT_SF_BUEN_MAL_100_U6M
        materialidad_externa = spark.table(
            "catalog_lhcl_prod_bcp.bcp_ddv_rbmbcapym_apppyme_vu.hm_variablerccmaterialidadapppyme"
        ).filter(F.col("codmes") == mes_data).select(
            F.col("CODMES").alias("CODMES_0"),
            F.col("codclaveunicocli"),
            F.col("RCC_TIP_COND_MOR_MAX_CRNNR_100_MAX_U12").alias("ATRASOMAX_CRNENR_12_100"),
            F.col("RCC_CTD_SF_CAL_CPP_FRQ_100_U24").alias("SF_NUM_CAL_CPP24_100"),
            F.col("RCC_CTD_MES_ACT_SF_BUEN_MAL_100_U6M").alias("MESES_ACTIVO_SF_BU_MAG_100")
        )
        materialidad_externa = materialidad_externa.withColumn('codmes', add_codmes_spark("CODMES_0", +1))
        materialidad_externa = materialidad_externa.drop("CODMES_0")

        # 4.4 Unir todas las fuentes
        carretera_completa = carretera_rcc.join(
            resumen_saldo, ["codmes", "codclaveunicocli"], "fullouter"
        )

        # FIX 3: Fullouter en vez de left outer
        carretera_con_mat = carretera_completa.join(
            materialidad_externa, ["codmes", "codclaveunicocli"], "fullouter"
        )

        # 4.5 Seleccionar columnas aplicando lógica de materialidad
        carretera_final = carretera_con_mat.select(
            "codmes",
            "codclaveunicoclí",
            F.when(F.col("ATRASOMAX_CRNENR_12_100").isNull(), F.col("ATRASOMAX_CRNENR_12"))
            .otherwise(F.col("ATRASOMAX_CRNENR_12_100")).alias("ATRASOMAX_CRNENR_12"),
            F.when(F.col("SF_NUM_CAL_CPP24_100").isNull(), F.col("SF_NUM_CAL_CPP24"))
            .otherwise(F.col("SF_NUM_CAL_CPP24_100")).alias("SF_NUM_CAL_CPP24"),
            # FIX 2: MESES_ACTIVO_SF_BU_MA6_100 de materialidad (se completa en 4.5b)
            F.col("MESES_ACTIVO_SF_BU_MA6_100").alias("MESES_ACTIVO_SF_BU_MA6_0_MAT"),
            "MONTOADE_ACT_MAX6_S_HIP",
            "UTL_3",
            "MESES_PMSAVMF_24_100"
        )

        # 4.5b Campo base MESES_ACTIVO de carretera (paso 11b del reference)
        carretera_meses_sf = spark.table(
            "catalog_lhcl_prod_bcp.bcp_ddv_matrizvariables_vu.hm_matrizdeudorrccproducto"
            ).filter(F.col("codmes") == mes_data).select(
            F.col("CODCMES").alias("CODCMES_0"),
            "codclaveunicoclí",
            F.col("RCC_CTD_MES_ACT_SF_BUEN_MAL_0_U6M").alias("MESES_ACTIVO_SF_BU_MA6_0_BASE")
            )
        carretera_meses_sf = carretera_meses_sf.withColumn("codmes", add_codmes_spark("CODCMES_0", +1))
        carretera_meses_sf = carretera_meses_sf.drop("CODCMES_0")

        carretera_final = carretera_final.join(
            carretera_meses_sf, ["codmes", "codclaveunicoclí"], "left_outer"
        )

        # Aplicar materialidad para MESES_ACTIVO_SF_BU_MA6_0
        carretera_final = carretera_final.withColumn(
            "MESES_ACTIVO_SF_BU_MA6_0",
            F.when(F.col("MESES_ACTIVO_SF_BU_MA6_0_MAT").isNull(), F.col("MESES_ACTIVO_SF_BU_MA6_0_BASE"))
            .otherwise(F.col("MESES_ACTIVO_SF_BU_MA6_0_MAT"))
        ).drop("MESES_ACTIVO_SF_BU_MA6_0_MAT", "MESES_ACTIVO_SF_BU_MA6_0_BASE")

        # ======================================================
        # 5. VARIABLES DEL DUENO (RL)
        # ======================================================
        logger.info("PASO 5: Variables del dueno (RL)")
        variables_dueno = relacion_dueno_final.join(
            carretera_final.select(
            F.col("codmes"),
            F.col("codclaveunicoclí").alias("CODCLAVEUNICOCLIREL"),
            F.col("ATRASOMAX_CRNENR_12").alias("ATRASOMAX_CRNENR_12_RL"),
            F.col("MONTOADE_ACT_MAX6_S_HIP").alias("MONTOADE_ACT_MAX6_S_HIP_RL"),
            F.col("UTL_3").alias("UTL_3_RL"),
            F.col("SF_NUM_CAL_CPP24").alias("SF_NUM_CAL_CPP24_RL"),
            F.col("MESES_ACTIVO_SF_BU_MA6_0").alias("MESES_ACTIVO_SF_BU_MA6_0_RL"),
            F.col("MESES_PMSAVMF_24_100").alias("MESES_PMSAVMF_24_100_RL")
            ),
            ["CODCLAVEUNICOCLIREL", "codmes"],
            "left_outer"
            ).select(
            "codmes", "CODCLAVEUNICOCLÍ",
            "ATRASOMAX_CRNENR_12_RL", "MONTOADE_ACT_MAX6_S_HIP_RL", "UTL_3_RL",
            "SF_NUM_CAL_CPP24_RL", "MESES_ACTIVO_SF_BU_MA6_0_RL", "MESES_PMSAVMF_24_100_RL"
            ).dropDuplicates(["codmes", "CODCLAVEUNICOCLÍ"])

        #====================================================
        #  6. VARIABLES AGREGADAS (_AG)
        # ======================================================
        logger.info("PASO 6: Variables agregadas cliente/dueño (_AG)")
        variables_cliente = carretera_final.select(
            "codmes", "codclaveunicoclí", "SF_NUM_CAL_CPP24", "MESES_ACTIVO_SF_BU_MA6_0"
        )

        resultado = universo.join(
            universo_apppyme, ["codmes", "CODCLAVEUNICOCLI"], "left_outer"
        ).join(
            variables_dueno, ["codmes", "CODCLAVEUNICOCLI"], "left_outer"
            ).join(
            variables_cliente, ["codmes", "codclaveunicoclí"], "left_outer"
        )

        resultado = resultado.withColumn(
            "SF_NUM_CAL_CPP24_AG_raw",
            operacionesMaxBetweenCols_udf(array(F.col("SF_NUM_CAL_CPP24"), F.col("SF_NUM_CAL_CPP24_RL")))
        ).withColumn(
            "SF_NUM_CAL_CPP24_AG",
            F.when(F.col("TIPPARTYIDENTIFICACION") != '6', F.col("SF_NUM_CAL_CPP24"))
            .otherwise(F.col("SF_NUM_CAL_CPP24_AG_raw"))
            .cast(IntegerType())
        )

        resultado = resultado.withColumn(
            "MESES_ACTIVO_SF_BU_MA6_0_AG_raw",
            operacionesMaxBetweenCols_udf(array(F.col("MESES_ACTIVO_SF_BU_MA6_0"), F.col("MESES_ACTIVO_SF_BU_MA6_0_RL")))
        ).withColumn(
            "MESES_ACTIVO_SF_BU_MA6_0_AG",
            F.when(F.col("TIPPARTYIDENTIFICACION") != '6', F.col("MESES_ACTIVO_SF_BU_MA6_0"))
            .otherwise(F.col("MESES_ACTIVO_SF_BU_MA6_0_AG_raw"))
            .cast(IntegerType())
        )

        #====================================================
        # 7. VARIABLES ADICIONALES: ANTIGUEDAD SUNAT (desfase +1)
        #====================================================
        logger.info("PASO 7: Variables adicionales (antiguedad SUNAT)")
        contribuyente_sunat = spark.table(
            "catalog_lhcl_prod_bcp.bcp_ddv_rbmbcapym_modelogestion_vu.hm_contribuyentesunatpyme"
        ).filter(F.col("codmes") == mes_data).select(
            F.col("CODMEES").alias("CODMEES_0"),
            F.col("CODCLAVEUNICOCCI"),
            F.col("CTDMESANTIGUEDADEMP").cast(IntegerType()).alias("ctdmesantiguedadempsunat")
        )
        contribuyente_sunat = contribuyente_sunat.withColumn('codmes', add_codmes_spark('CODMES_0', +1))
        contribuyente_sunat = contribuyente_sunat.drop("CODMES_0")
        resultado = resultado.join(contribuyente_sunat, ["codclaveunicoclí", "codmes"], "left_outer")

        #====================================================
        # 8. SELECCIÓN FINAL Y CAST
        #====================================================
       resultado_final = resultado.select(
            F.col("codmes").alias("codmes"),
            F.col("codclaveunicoclí").alias("codclaveunicoclí"),
            F.col("codclavepartycli").alias("codclavepartycli"),
            F.col("codinternocomputacional").alias("codinternocomputacional"),
            F.col("EDAD_FIN").cast(IntegerType()).alias("edad_fin"),
            F.col("ACT_ECO_FIN").alias("act_eco_fin"),
            F.col("ATRASOMAX_CRNENR_12_RL").cast(IntegerType()).alias("atrasomax_crnenr_12_rl"),
            F.col("MONTOADE_ACT_MAX6_S_HIP_RL").cast(DecimalType(19, 8)).alias("montoade_act_max6_s_hip_rl"),
            F.col("UTL_3_RL").cast(DecimalType(23, 6)).alias("utl_3_rl"),
            F.col("MESES_PMSAVMF_24_100_RL").cast(IntegerType()).alias("meses_pmsavmf_24_100_rl"),
            F.col("SF_NUM_CAL_CPP24_AG").cast(IntegerType()).alias("sf_num_cal_cpp24_ag"),
            F.col("MESES_ACTIVO_SF_BU_MA6_0_AG").cast(IntegerType()).alias("meses_activo_sf_bu_ma6_0_ag"),
            F.col("ctdmesantiguedadempssunat").cast(IntegerType()).alias("ctdmesantiguedadempssunat")
        ).distinct()

        #====================================================
        # 8. VARIABLES CON DESFASE +1 (tablas VIDEA)
        # ===================================================
        logger.info("PASO 8: Variables VIDA con desfase +1")

        VIDEVAR_MTX_MORA = spark.table(
            "catalog_lhcl_prod_bcp.bcp_ddv_adrmmgr_videavariablesmodelos_vu.hm_matrizmoraponderadaclientemmgr"
        ).filter(F.col("codes") == mes_data).select(
            F.col("CODES").alias("CODES_0"), "codlaveunicoclí",
            F.col("mtodeudaclasifriesgofactordscotsoalu12").alias(           "VIDEVAR_MTX_MORA_POND_CLI_MMGR_mtodeudaclasifriesgofactordscotsoalu12")
        )
        VIDEVAR_MTX_MORA = VIDEVAR_MTX_MORA.withColumn("codes", add_codes_spark("CODES_0", +1)).drop("CODES_0")

        PASIVO_EVOL = spark.table(
            "catalog_lhcl_prod_bcp.bcp_ddv_adrmmgr_videavariablesmodelos_vu.hm_variablepasivoevolucionsaldopyme"
        ).filter(F.col("codes") == mes_data).select(
            F.col("CODES").alias("CODES_0"), "codinternocomputacional",
            F.col("mtoprmincrvariacionmensualprmvigsolun6m").alias("PASIVO_EVOL_SALD_PVM_mtoprmincrvariacionmensualprmvigsolun6m")
        )
        PASIVO_EVOL = PASIVO_EVOL.withColumn("codes", add_codes_spark("CODES_0", +1)).drop("CODES_0")

        #====================================================
        # 9. VARIABLES SIN DESFASE (codmes actual)
        #====================================================
        logger.info("PASO 9: Variables de matrices (sin desfase)")

        MTX_RCC_OTRA = spark.table(
            "catalog_lhcl_prod_bcp_bcp_ddv_matrizvariables_vu.hm_matrizdeudorrcctardeuda"
        ).filter(F.col("codmes") == mes_actual_proceso).select(
            "codmes", "codclaveunicoclí",
            F.col("rcc_mto_rdv_max_u3m").alias("MTX_RCC_OTRA_DEUDA_rcc_mto_rdv_max_u3m")
        ).distinct()

        CLASI_EXPER = spark.table(
            "catalog_lhcl_prod_bcp_bcp_ddv_adrmngr_videavariablesmodelos_vu.hm_clasificacionclientenivelexperienciapyme"
        ).filter(F.col("codmes") == mes_actual_proceso).select(
            "codmes", "codclaveunicoclí",
            F.col("ctdempleado").alias("CLASI_EXPER_CLI_ctdempleado")
        ).distinct()

        EVOL_CLI = spark.table(
            "catalog_lhcl_prod_bcp_bcp_ddv_adrmngr_videavariablesmodelos_vu.hm_variableactivovevolucionclientepyme"
        ).filter(F.col("codmes") == mes_actual_proceso).select(
            "codmes", "codinternocomputacional",
            F.col("ctddiamoramay8u6m").alias("EVOL_CLI_PYM_ctddiamoramay8u6m"),
            F.col("ctdmaxdiamarau6m").alias("EVOL_CLI_PYM_ctdmaxdiamarau6m")
        ).distinct()

        MTX_RESUMEN_SALDO_FULL = spark.table(
            "catalog_lhcl_prod_bcp_bcp_ddv_matrizvariables_vu.hm_matrizresumensaldo"
        ).filter(F.col("codmes") == mes_actual_proceso).select(
            "codmes", "codclaveunicoclí",
            F.col("prod_antmin_cvi_prm_u12").alias("MTX_RESUMEN_SALDO_prod_antmin_cvi_prm_u12"),
            F.col("prod_ctd_pai_max_u6m").alias("MTX_RESUMEN_SALDO_prod_ctd_pai_max_u6m"),
            F.col("prod_ctd_sld_act_u1m").alias("MTX_RESUMEN_SALDO_prod_ctd_sld_act_u1m"),
            F.col("prod_mto_sld_pai_g6m").alias("MTX_RESUMEN_SALDO_prod_mto_sld_pai_g6m")
        ).distinct()

        MTX_RESUMEN_ACT_PAS = spark.table(
            "catalog_lhcl_prod_bcp.bcp_ddv_matrizvariables_vu.hm_matrizresumensaldoactivopasivo"
        ).filter(F.col("codmes") == mes_actual_proceso).select(
            "codmes", "codclaveunicocli",
            F.col("prod_ctd_mes_pas_act_max_v2_0_u12").alias("MTX_RESUMEN_ACT_PAS__prod_ctd_mes_pas_act_max_v2_0_u12"),
            F.col("prod_mto_sld_fim_pas_min_24_24_rt_u24").alias("MTX_RESUMEN_ACT_PAS__prod_mto_sld_fim_pas_min_24_24_rt_u24"),
            F.col("prod_mto_sld_fim_tsav_max_12_12_rt_u12").alias("MTX_RESUMEN_ACT_PAS__prod_mto_sld_fim_tsav_max_12_12_rt_u12"),
            F.col("prod_mto_sld_fim_tsav_med_1_6_rt_u6m").alias("MTX_RESUMEN_ACT_PAS__prod_mto_sld_fim_tsav_med_1_6_rt_u6m"),
            F.col("prod_mto_sld_prm_tsav_max_min_6_6_rt_u6m").alias("MTX_RESUMEN_ACT_PAS__prod_mto_sld_prm_tsav_max_min_6_6_rt_u6m"),
            F.col("prod_mto_sld_prm_tsav_med_1_6_rt_u6m").alias("MTX_RESUMEN_ACT_PAS__prod_mto_sld_prm_tsav_med_1_6_rt_u6m"),
            F.col("prod_pct_fm_pmpas_med_12_12_rt_u12").alias("MTX_RESUMEN_ACT_PAS__prod_pct_fm_pmpas_med_12_12_rt_u12"),
            F.col("prod_pct_pmtsav_pmact_24_24_rt_u24").alias("MTX_RESUMEN_ACT_PAS__prod_pct_pmtsav_pmact_24_24_rt_u24"),
            F.col("prod_mto_sld_prm_pas_max_min_12_12_rt_u12").alias("MTX_RESUMEN_ACT_PAS__prod_mto_sld_prm_pas_max_min_12_12_rt_u12"),
            F.col("prod_mto_sld_fim_act_min_6_6_rt_u6m").alias("MTX_RESUMEN_ACT_PAS__prod_mto_sld_fim_act_min_6_6_rt_u6m"),
            F.col("prod_mto_sld_prm_act_max_min_6_6_rt_u6m").alias("MTX_RESUMEN_ACT_PAS__prod_mto_sld_prm_act_max_min_6_6_rt_u6m"),
            F.col("prod_mto_sld_fim_tsav_max_min_12_12_rt_u12").alias("MTX_RESUMEN_ACT_PAS__prod_mto_sld_fim_tsav_max_min_12_12_rt_u12"),
            F.col("prod_mto_sld_fim_act_max_min_6_6_rt_u6m").alias("MTX_RESUMEN_ACT_PAS__prod_mto_sld_fim_act_max_min_6_6_rt_u6m"),
        ).distinct()

        MTX_MOV_ABONO = spark.table(
            "catalog_lhcl_prod.bcp.bcp_ddv_matrizvariables_vu.hm_matrizmovimientoabonopasivo"
        ).filter(F.col("codmes") == mes_actual_proceso).select(
            "codmes", "codclaveunicocli",
            F.col("isav_flg_opea_nhab_ncts_dol_max_u12").alias("MTX_MOV_ABONO_PAS__isav_flg_opea_nhab_ncts_dol_max_u12"),
            F.col("isav_tkt_opea_trnf_dol_max_u3m").alias("MTX_MOV_ABONO_PAS__isav_tkt_opea_trnf_dol_max_u3m"),
            F.col("isav_tkt_opea_trnf_min_u9m").alias("MTX_MOV_ABONO_PAS__isav_tkt_opea_trnf_min_u9m")
        ).distinct()

        MTX_MOV_CARGO = spark.table(
            "catalog_lhcl_prod.bcp.bcp_ddv_matrizvariables_vu.hm_matrizmovimientocargopasivo"
        ).filter(F.col("codmes") == mes_actual_proceso).select(
            "codmes", "codclaveunicocli",
            F.col("isav_mto_opec_retr_g6m").alias("MTX_MOV_CARGO_PAS__isav_mto_opec_retr_g6m"),
            F.col("isav_tkt_opec_pago_srv_prm_u3m").alias("MTX_MOV_CARGO_PAS__isav_tkt_opec_pago_srv_prm_u3m")
        ).distinct()

        MTX_MOV_PAS = spark.table(
            "catalog_lhcl_prod.bcp.bcp_ddv_matrizvariables_vu.hm_matrizmovimientopasivo"
        ).filter(F.col("codmes") == mes_actual_proceso).select(
            "codmes", "codclaveunicocli",
            F.col("isav_tkt_opec_sav_min_u9m").alias("MTX_MOV_PAS__isav_tkt_opec_sav_min_u9m")
        ).distinct()

        MTX_TRX_PAGO = spark.table(
            "catalog_lhcl_prod.bcp.bcp_ddv_matrizvariables_vu.hm_matriztransaccioncanalpago transferencia"
        ).filter(F.col("codmes") == mes_actual_proceso).select(
            "codmes", "codclaveunicocli",
            F.col("can_ctd_tmo_tot_pag_bcp_frq_u6m").alias("MTX_TRX_CANAL_PAGO_TRANSF__can_ctd_tmo_tot_pag_bcp_frq_u6m"),
            F.col("can_mto_tmo_tot_pag_bcp_prm_u6m").alias("MTX_TRX_CANAL_PAGO_TRANSF__can_mto_tmo_tot_pag_bcp_prm_u6m"),
            F.col("can_tkt_tmo_tot_pag_srv_g6m").alias("MTX_TRX_CANAL_PAGO_TRANSF__can_tkt_tmo_tot_pag_srv_g6m"),
            F.col("can_tkt_tmo_tot_pag_srv_sol_g6m").alias("MTX_TRX_CANAL_PAGO_TRANSF__can_tkt_tmo_tot_pag_srv_sol_g6m")
        ).distinct()

        MTX_TRX_POS = spark.table(
            "catalog_lhcl_prod.bcp.bcp_ddv_matrizvariables_vu.hm_matriztransaccionpos"
        ).filter(F.col("codmes") == mes_actual_proceso).select(
            "codmes", "codclaveunicocli",
            F.col("pos_tkt_trx_com_sol_prm_p6m").alias("MTX_TRX_POS__pos_tkt_trx_com_sol_prm_p6m"),
            F.col("pos_tkt_trx_td_prm_u6m").alias("MTX_TRX_POS__pos_tkt_trx_td_prm_u6m")
        ).distinct()

        MTX_RCC_PROD = spark.table(
            "catalog_lhcl_prod.bcp.bcp_ddv_matrizvariables_vu.hm_matrizdeudorrcproducto"
        ).filter(F.col("codmes") == mes_actual_proceso).select(
            "codmes", "codclaveunicocli",
            F.col("rcc_ctd_mes_act_sf_buen_1000_frq_v_u12").alias("MTX_RCC_PROD__rcc_ctd_mes_act_sf_buen_1000_frq_v_u12"),
            F.col("rcc_pct_deu_cpp_max_u12").alias("MTX_RCC_PROD__rcc_pct_deu_cpp_max_u12"),
            F.col("rcc_pct_deu_defc_max_u6m").alias("MTX_RCC_PROD__rcc_pct_deu_defc_max_u6m"),
            F.col("rcc_tip_cond_mor_max_crnor_max_u6m").alias("MTX_RCC_PROD__rcc_tip_cond_mor_max_crnor_max_u6m")
        ).distinct()

        MTX_TRX_CANAL = spark.table(
            "catalog_lhcl_prod.bcp.bcp_ddv_matrizvariables_vu.hm_matriztransaccioncanal"
        ).filter(F.col("codmes") == mes_actual_proceso).select(
            "codmes", "codclaveunicocli",
            F.col("can_tkt_tmo_tot_ret_sol_max_u6m").alias("MTX_TRX_CANAL__can_tkt_tmo_tot_ret_sol_max_u6m"),
            F.col("can_tkt_tmo_tot_sol_min_u12").alias("MTX_TRX_CANAL__can_tkt_tmo_tot_sol_min_u12")
        ).distinct()

        # ===================================
        # 10. MODULO ACTIVO (lógica compleja)
        # ===================================
        logger.info("PASO 10: Módulo Activo")
        from pyspark.sql.window import Window as W2
        from dateutil.relativedelta import relativedelta
        from datetime import datetime

        codmescuatro = int(month_delta(str(mes_actual_proceso), -4))
        codmestres = int(month_delta(str(mes_actual_proceso), -3))

        def generar_meses(mes_inicio, mes_fin):
            meses = []
            fecha_fin = datetime.strptime(str(mes_fin), "%Y%m")
            fecha_actual = datetime.strptime(str(mes_inicio), "%Y%m")
            while fecha_actual <= fecha_fin:
                meses.append(fecha_actual.strftime("%Y%m"))
                fecha_actual += relativedelta(months=1)
            return meses

        # Universo segmento: solo clientes con flgactmay100may6u12 = 1
        universo_segmento = spark.table(
            "catalog_lhcl_prod.bcp.bcp_ddv_rbmbcapym_segmentacionpyme_vu.hm_segmentoinformativo pyme"
        ).filter(
            (F.col("codmes") == mes_data) & (F.col("flgactmay100may6u12") == 1)
        ).select("CODCLAVEUNICOCLI").distinct()

        # Transacciones
        transacciones = spark.table(
            "catalog_lhcl_prod.bcp.bcp_ddv_rbmbcapym_apppyme_vu.hm_matrizbasepasivoclienteapppyme"
        ).filter(F.col("codmes") == mes_data).select(
            F.col("CODCLAVEPARTYCLI"),
            (F.col("ISAV_MTO_OPEA_ESTVTA_PYM_PRM_U6M") / F.col("ISAV_MTO_OPEA_ESTVTA_PYM_MAX_U12")).alias("isav_mto_opea_estvta_pym_u6m_rt_max_u12"),
            (F.col("ISAV_MTO_OPEA_ESTVTA_PYM_PRM_U6M") / F.col("ISAV_MTO_OPEC_ESTVTA_PYM_PRM_U12")).alias("pctratiomtoopeaprmu6mopecprmu12")
        )

        # Portafolio para variación de deuda
        portafolio_act = spark.table(
            "catalog_lhcl_prod.bcp.bcp_ddv_adrmmgr_seginfobasesgenerales_vu.hm_portafoliocredito"
        ).filter(
            (F.col("CODMES").between(codmescuatro, mes_data)) &
            (F.trim(F.col("FLGCTAVALIDA")) == "1") &
            (F.col("TIPESTADOCCTA").isin("A", "AC", "D"))
        ).select("CODCLAVEPARTYCLI", "MTOSALDOCAPITALSOL", "CODMES")

        portafolio_agrupado = portafolio_act.groupBy("CODCLAVEPARTYCLI", "CODMES").agg(
            F.sum("MTOSALDOCAPITALSOL").alias("mtosaldocapitalsol_tmp")
        )

        lista_meses = generar_meses(codmescuatro, mes_data)
        df_meses = spark.createDataFrame(lista_meses, StringType()).select(F.col("value").alias("CODMES"))
        partys = portafolio_agrupado.select("CODCLAVEPARTYCLI").dropDuplicates()
        cross = partys.crossJoin(F.broadcast(df_meses))
        portafolio_completo = portafolio_agrupado.join(cross, on=["CODMES", "CODCLAVEPARTYCLI"], how="right_outer")

        w = W2.PartitionBy("CODCLAVEPARTYCLI").orderBy(F.col("CODMES").desc())
        portafolio_var = portafolio_completo.withColumn(
            "mtosaldocapitalsol_tmp_prev", F.lead("mtosaldocapitalsol_tmp", 1).over(w).cast("double")
        ).withColumn(
            "mtosaldocapitalsol_tmp_VAR_DEC",
            F.when(F.col("mtosaldocapitalsol_tmp").isNull() & F.col("mtosaldocapitalsol_tmp_prev").isNull(), F.lit(None))
            .otherwise(
                F.when(F.coalesce(F.col("mtosaldocapitalsol_tmp"), F.lit(0)) - F.coalesce(F.col("mtosaldocapitalsol_tmp_prev"), F.lit(0)) > 0, F.lit(0))
                .otherwise(F.abs(F.coalesce(F.col("mtosaldocapitalsol_tmp"), F.lit(0)) - F.coalesce(F.col("mtosaldocapitalsol_tmp_prev"), F.lit(0))))
            )
        )

        dec_var_prm3 = portafolio_var.filter(
            F.col("CODMES").between(str(codmestres), str(mes_data))
        ).groupBy("CODCLAVEPARTYCLI").agg(
            F.avg("mtosaldocapitalsol_tmp_VAR_DEC").alias("dec_var_MONTO_meanPrev3")
        )

        pasivos = spark.table(
            "catalog_lhcl_prod.bcp.bcp_ddv_rbmbcapym_apppyme_vu.hm_matrizbaseresumensaldoapppyme"
        ).filter(F.col("codmes") == mes_data).select(
            "CODCLAVEUNICOCLI", "PROD_MTO_SLD_PRM_VIG_PAS_AHO_CTEORD_PRM_U3M"
        )

        # Unir con universo del mes actual
        bd_mod_act = universo.join(universo_segmento.withColumn("en_universo", F.lit(1)), "CODCLAVEUNICOCLI", "left")
        bd_mod_act = bd_mod_act.join(transacciones, "CODCLAVEPARTYCLI", "left").join(dec_var_prm3, "CODCLAVEPARTYCLI", "left").join(pasivos, "CODCLAVEUNICOCLI", "left")

        bd_mod_act = bd_mod_act.withColumn(
            "pctratiomtodecdeudapymer tmtopasivoprmu3m_raw",
            F.col("dec_var_MONTO_meanPrev3") / F.col("PROD_MTO_SLD_PRM_VIG_PAS_AHO_CTEORD_PRM_U3M")
        )

        # Si NO está en universo segmento -> NULL
        bd_mod_act = bd_mod_act.withColumn(
            "MOD_ACT__isav_mto_opea_estvta_pym_u6m_rt_max_u12",
            F.when(F.col("en_universo") == 1, F.col("isav_mto_opea_estvta_pym_u6m_rt_max_u12")).otherwise(F.lit(None))
        ).withColumn(
            "MOD_ACT__pctratiomtodecdeudapymer tmtopasivoprmu3m",
            F.when(F.col("en_universo") == 1, F.col("pctratiomtodecdeudapymer tmtopasivoprmu3m_raw")).otherwise(F.lit(None))
        ).withColumn(
            "MOD_ACT__pctratiomtoopeaprmu6mopecprmu12",
            F.when(F.col("en_universo") == 1, F.col("pctratiomtoopeaprmu6mopecprmu12")).otherwise(F.lit(None))
        )

        resultado_mod_act = bd_mod_act.select(
            "codmes", "codclaveunicocli",
            F.col("MOD_ACT__isav_mto_opea_estvta_pym_u6m_rt_max_u12").cast(DecimalType(19, 8)),
            F.col("MOD_ACT__pctratiomtodecdeudapymer tmtopasivoprmu3m").cast(DecimalType(19, 8)),
            F.col("MOD_ACT__pctratiomtoopeaprmu6mopecprmu12").cast(DecimalType(19, 8))
        )

        # ========================================
        # 11. JOIN TODAS LAS VARIABLES AL UNIVERSO
        # ========================================
        logger.info("PASO 11: Unir todas las variables")

        # Renombrar las vars ya construidas con prefijo APP_SCORE_APROB_PYME__
        resultado_score = resultado_final.select(
            "codmes", "codclaveunicocli", "codclavepartycli", "codinternocomputacional",
            F.col("edad_fin").alias("APP_SCORE_APROB_PYME__edad_fin"),
            F.col("act_eco_fin").alias("APP_SCORE_APROB_PYME__act_eco_fin"),
            F.col("atrasomax_crnenr_12_rl").alias("APP_SCORE_APROB_PYME__atrasomax_crnenr_12_rl"),
            F.col("montoade_act_max6_s_hip_rl").alias("APP_SCORE_APROB_PYME__montoade_act_max6_s_hip_rl"),
            F.col("utl_3_rl").alias("APP_SCORE_APROB_PYME__utl_3_rl"),
            F.col("meses_pmsavmf_24_100_rl").alias("APP_SCORE_APROB_PYME__meses_pmsavmf_24_100_rl"),
            F.col("sf_num_cal_cpp24_ag").alias("APP_SCORE_APROB_PYME__sf_num_cal_cpp24_ag"),
            F.col("meses_activo_sf_bu_ma6_0_ag").alias("APP_SCORE_APROB_PYME__meses_activo_sf_bu_ma6_0_ag"),
            F.col("ctdmesantiguedademsunat").alias("MOD_DEMO__ctdmesantiguedademsunat")
        )

        # Joins por codclaveunicocli
        joins_codclave = [
            resultado_mod_act, VIDEVAR_MTX_MORA,
            MTX_RCC_OTRA, CLASI_EXPER,
            MTX_RESUMEN_SALDO_FULL, MTX_RESUMEN_ACT_PAS,
            MTX_MOV_ABONO, MTX_MOV_CARGO, MTX_MOV_PAS,
            MTX_TRX_PAGO, MTX_TRX_POS, MTX_RCC_PROD, MTX_TRX_CANAL
        ]
        base = resultado_score
        for tabla in joins_codclave:
            cols_join = ["codmes", "codclaveunicocli"]
            extra = [c for c in tabla.columns if c not in cols_join]
            base = base.join(tabla.select(*cols_join, *extra), on=cols_join, how="left")

        # Joins por codinternocomputacional
        joins_codinterno = [PASIVO_EVOL, EVOL_CLI]
        for tabla in joins_codinterno:
            cols_join = ["codmes", "codinternocomputacional"]
            extra = [c for c in tabla.columns if c not in cols_join]
            base = base.join(tabla.select(*cols_join, *extra), on=cols_join, how="left")

        # =====================
        # 12. DUMMY REPLACEMENT
        # =====================
        logger.info("PASO 12: Reemplazo de valores dummy")
        GLOB_DUMMY_LIST = [1111111111, -1111111111, 2222222222, -2222222222, 3333333333,
                        -3333333333, 4444444444, 5555555555, 6666666666, 7777777777, -99, -999,
                        44444.4444, 55555.5555, 66666.6666, 77777.7777, 11111.1111,
                        -11111.1111, 22222.2222, -22222.2222, 33333.3333, -33333.3333]
        cols_all = base.columns
        base = base.select(*[
            F.when(F.col(c).isin(GLOB_DUMMY_LIST), None).otherwise(F.col(c)).alias(c) for c in cols_all
        ])

        # ========================
        # 13. RNG_ACTIVIDAD_ECONOM
        # ========================
        logger.info("PASO 13: Codificación actividad económica")
        base = base.withColumn("RNG_ACTIVIDAD_ECONOM",
            F.when(F.col("APP_SCORE_APROB_PYME__act_eco_fin").isNull(), 1)
            .when(F.col("APP_SCORE_APROB_PYME__act_eco_fin").isin(
                'PESCA', 'OTROS', 'SERVICIOS', 'ENERGIA', 'CONSTRUCCION', 'ADM_PUBLICA',
                'ACT_INMOB', 'EMP_Y_DE_ALQ', 'INDUST_MANUFACT', 'COMERCIO', 'HOGAR', 'SALUD'), 1)
            .otherwise(0)
        )

        # ==================================
        # 14. CAPPING (crear variables _cap)
        # ==================================
        logger.info("PASO 14: Capping de variables")
        def cap(df, col, lower=None, upper=None):
            out = col + "_cap"
            expr = F.col(col)
            if lower is not None:
                expr = F.greatest(expr, F.lit(float(lower)))
            if upper is not None:
                expr = F.least(expr, F.lit(float(upper)))
            return df.withColumn(out, expr)

        base = cap(base, "MTX_RCC_OTRA_DEUDA__rcc_mto_rdv_max_u3m", upper=54058.633)
        base = cap(base, "CLASI_EXPER_CLI__ctdempleado", lower=0, upper=203.0)
        base = cap(base, "MTX_RESUMEN_ACT_PAS__prod_mto_sld_prm_pas_max_min_12_12_rt_u12", upper=88040.516)
        base = cap(base, "EVOL_CLI_PYM__ctdmaxdiamora u6m", lower=0, upper=59.0)
        base = cap(base, "MTX_RESUMEN_SALDO__prod_ctd_sld_act_u1m", upper=13.0)
        base = cap(base, "MOD_DEMO__ctdmesantiguedademsunat", upper=396.0)
        base = cap(base, "APP_SCORE_APROB_PYME__utl_3_rl", lower=0, upper=261.9096)
        base = cap(base, "MTX_RESUMEN_ACT_PAS__prod_mto_sld_fim_pas_min_24_24_rt_u24", upper=0.8768666)
        base = cap(base, "MTX_TRX_CANAL__can_tkt_tmo_tot_sol_min_u12", upper=28916.992)
        base = cap(base, "MTX_RESUMEN_ACT_PAS__prod_pct_pmtsav_pmact_24_24_rt_u24", upper=19.579279)
        base = cap(base, "MTX_RESUMEN_ACT_PAS__prod_mto_sld_prm_tsav_max_min_6_6_rt_u6m", upper=4771.6816)
        base = cap(base, "MTX_RESUMEN_ACT_PAS__prod_mto_sld_prm_tsav_med_1_6_rt_u6m", upper=4.0419364)
        base = cap(base, "MOD_ACT__pctratiomtodecdeudapymer tmtopasivoprmu3m", upper=480.26755)
        base = cap(base, "APP_SCORE_APROB_PYME__edad_fin", lower=25, upper=75)
        base = cap(base, "PASIVO_EVOL_SALD_PYM__mtoprm incrvariacionmensualprm vigsolu6m", upper=211663.56)
        base = cap(base, "MTX_RESUMEN_ACT_PAS__prod_mto_sld_prm_act_max_min_6_6_rt_u6m", upper=386732.84)
        base = cap(base, "MTX_MOV_ABONO_PAS__isav_tkt_opea_trnf_min_u9m", upper=46430.434)
        base = cap(base, "MTX_RCC_PROD__rcc_tip_cond_mor_max_crnor_max_u6m", upper=469.555)
        base = cap(base, "VIDEVAR_MTX_MORA_POND_CLI_NMGR__mtodeudaclasif iesgofactordsctosolu12", upper=709515.0)
        base = cap(base, "MOD_ACT__isav_mto_opea_estvta_pym_u6m_rt_max_u12", lower=0, upper=1)
        base = cap(base, "MTX_RESUMEN_ACT_PAS__prod_mto_sld_fim_tsav_max_min_12_12_rt_u12", upper=372685.25)
        base = cap(base, "MTX_RESUMEN_SALDO__prod_antmin_cvi_prm_u12", upper=318.5)
        base = cap(base, "MTX_RESUMEN_SALDO__prod_mto_sld_pai_g6m", upper=58.04616)
        base = cap(base, "MTX_MOV_ABONO_PAS__isav_tkt_opea_trnf_dol_max_u3m", upper=379400.0)
        base = cap(base, "MOD_ACT__pctratiomtoopeaprmu6mopecprmu12", upper=2.637802)
        base = cap(base, "MTX_TRX_POS__pos_tkt_trx_td_prm_u6m", lower=0, upper=6189.8115)
        base = cap(base, "MTX_TRX_CANAL_PAGO_TRANSF__can_mto_tmo_tot_pag_bcp_prm_u6m", upper=112905.56)
        base = cap(base, "MTX_RESUMEN_ACT_PAS__prod_pct_fm_pmpas_med_12_12_rt_u12", upper=3.8366303)
        base = cap(base, "MTX_TRX_POS__pos_tkt_trx_com_sol_prm_p6m", lower=0, upper=7833.3335)
        base = cap(base, "MTX_TRX_CANAL_PAGO_TRANSF__can_tkt_tmo_tot_pag_srv_g6m", upper=133.00755)
        base = cap(base, "APP_SCORE_APROB_PYME__atrasomax_crnenr_12_rl", upper=46.0)

        # =========================================================
        # 15. SELECCIÓN FINAL - resultado_final con todas las features
        # =========================================================
        logger.info("PASO 15: Selección final de features + IDs")
        resultado_final = base.select(
            "codmes", "codclaveunicocli", "codclavepartycli", "codinternocomputacional",
            *[F.col(f).cast("double").alias(f) for f in FEATURE_NAMES]
        ).distinct()

        # ========================================================
        # 9. ESCRITURA EN TABLA DELTA CON PARTICIONADO POR codmes
        # ========================================================
        from pyspark.sql import functions as F
        if 'codmes' in resultado_final.columns:
            sample = resultado_final.select(F.col("codmes")).first()[0]
            if isinstance(sample, str) and '-' in sample:
                resultado_final = resultado_final.withColumn(
                    "codmes",
                    (F.year(F.to_date(F.col("codmes"))) * 100 + F.month(F.to_date(F.col("codmes")))).cast("int")
                )
            else:
                resultado_final = resultado_final.withColumn("codmes", F.col("codmes").cast("int"))
        else:
            raise ValueError("La columna 'codmes' no existe en resultado_final")

        if idx_mes == 1:
            write_mode = "overwrite"
            logger.info(f"Primer mes ({mes_actual_proceso}) - Modo: OVERWRITE")
        else:
            write_mode = "append"
            logger.info(f"Mes {mes_actual_proceso} - Modo: APPEND (agregando partición)")

        resultado_final.write.mode(write_mode) \
            .option("overwriteSchema", "true") \
            .partitionBy("codmes") \
            .saveAsTable(OUTPUT_FEATURES_TABLE)

        count_final = resultado_final.count()
        logger.info(f"Registros escritos para mes {mes_actual_proceso}: {count_final:,}")
        logger.info(f"Tabla actualizada: {OUTPUT_FEATURES_TABLE}")

        total_duration = time.time() - start_time
        logger.log_stage_end('load_preparation_data', duration=total_duration, records_extracted=count_final)
        resultados.append({
            'mes': mes_actual_proceso,
            'registros': count_final,
            'exitoso': True,
            'duracion': total_duration
        })

    except Exception as e:
        logger.log_exception(
            operation='load_preparation_data',
            exception=e,
            should_raise=False,
            mes_vta=mes_actual_proceso
        )
        resultados.append({
            'mes': mes_actual_proceso,
            'registros': 0,
            'exitoso': False,
            'error': str(e)
        })
        continue

In [0]:
# Se muestra el resultado parcial en caso de que en la celda Código Load Preparation Data haya
# algún error de escritura en alguna de las taxonomías, # nomenclatura de tablas y/o columnas.
print("=== DIAGNÓSTICO PREVIO A ESCRITURA ===")
print(f"resultado_final vacío? {resultado_final.count() == 0}")
print("Esquema:")
resultado_final.printSchema()
print("Muestra:")
resultado_final.limit(3).display()

In [0]:
# ================================
# VERIFICACIÓN DE LA TABLA CREADA
# ================================
logger.info("Verificando tabla generada...")

# 1. Existencia y conteo
df_check = spark.sql(f"SELECT COUNT(*) as total FROM {OUTPUT_FEATURES_TABLE}")
total_registros = df_check.collect()[0]['total']
print(f"✅ Tabla {OUTPUT_FEATURES_TABLE} existe con {total_registros:,} registros totales.")

# 2. Verificar particiones (mostrar lista y conteo)
particiones_df = spark.sql(f"SHOW PARTITIONS {OUTPUT_FEATURES_TABLE}")
particiones_count = particiones_df.count()
print(f"✅ Particiones detectadas: {particiones_count}")
print("Lista de particiones:")
display(particiones_df)

# 3. Tomar una muestra
print("Muestra de 3 registros:")
display(spark.sql(f"SELECT * FROM {OUTPUT_FEATURES_TABLE} LIMIT 3"))

# 4. Guardar métricas en task values
dbutils.jobs.taskValues.set(key="features_table", value=OUTPUT_FEATURES_TABLE)
dbutils.jobs.taskValues.set(key="features_total_records", value=total_registros)
dbutils.jobs.taskValues.set(key="features_partitions", value=particiones_count)

In [0]:
# ==========================================================
# RESUMEN FINAL
# ==========================================================
logger.info("")
logger.info("=" * 80)
logger.info("RESUMEN FINAL")
logger.info("=" * 80)
exitosos = sum(1 for r in resultados if r['exitoso'])
logger.info(f"Meses: {len(resultados)} | OK: {exitosos} | Error: {len(resultados) - exitosos}")
for r in resultados:
    if r['exitoso']:
        logger.info(f" ✅ {r['mes']}: {r['registros']:,} registros en {r['duracion']:.2f}s")
    else:
        logger.error(f" ❌ {r['mes']}: {r.get('error', '?')}")

if 'logger' in locals():
    logger.info(f"Finalizando notebook: {logger.name}")
    logger._flush_all_handlers()
    logger.close()